# Amazon Categorizer Playground

Demonstrates the categorizer API against synthetic data.  No real Amazon CSV needed.
All outputs go to a temporary directory so nothing lands in `data/`, `out/`, or `state/`.

For real use replace `workspace` with the actual paths:
```python
state_dir = Path('state')
output_dir = Path('out')
input_csv  = Path('data/amazon_2026_01.csv')
```

In [12]:
import tempfile
from pathlib import Path

import pandas as pd

from analysis.amazon.notebook import (
    load_categorization_state,
    load_events_dataframe,
    run_amazon_categorization,
)
from analysis.log import configure_logging
from analysis.storage import JsonStateStore

# Isolated workspace for this notebook run
workspace = Path(tempfile.mkdtemp(prefix='amazon_playground_'))
state_dir  = workspace / 'state'
output_dir = workspace / 'out'
input_csv  = workspace / 'amazon_jan.csv'
configure_logging(state_dir=state_dir)

print('Workspace:', workspace)

Workspace: /var/folders/35/5769v36d35d8hm4dykc4ll_r0000gn/T/amazon_playground_tbv7hs_1


## Synthetic input CSV

Mix of items that will be pre-seeded as known and items that will be left uncategorized.

In [13]:
raw = pd.DataFrame({
    'Description': [
        'Echo Dot (3rd Gen)',
        'Paper Towels 12-Pack',
        'USB-C Cable 6ft',
        'Paper Towels 12-Pack',   # duplicate — should reuse mapping
        'Coffee Pods 36ct',
        'Unknown Gadget XYZ',
    ],
    'Amount': [24.99, 18.49, 9.99, 18.49, 28.00, 45.00],
})

raw.to_csv(input_csv, index=False)
raw

,Description,Amount
0,Echo Dot (3rd Gen),24.99
1,Paper Towels 12-Pack,18.49
2,USB-C Cable 6ft,9.99
3,Paper Towels 12-Pack,18.49
4,Coffee Pods 36ct,28.00
5,Unknown Gadget XYZ,45.00


## Pre-seed persistent memory

Simulates state carried over from previous months — the categorizer will auto-assign these without prompting.

In [14]:
store = JsonStateStore(state_dir)
store.save_categories(['Electronics', 'Household'])
store.save_mapping({
    'echo dot (3rd gen)':  'Electronics',
    'paper towels 12-pack': 'Household',
    'usb-c cable 6ft':     'Electronics',
})

cats, mapping = load_categorization_state(state_dir)
print('Pre-seeded categories:', cats)
print('Pre-seeded mappings:  ', mapping)

Pre-seeded categories: ['Electronics', 'Household']
Pre-seeded mappings:   {'echo dot (3rd gen)': 'Electronics', 'paper towels 12-pack': 'Household', 'usb-c cable 6ft': 'Electronics'}


## Run categorizer (non-interactive)

Unknown items are assigned `UNCATEGORIZED`.  In CLI use, `--interactive` prompts for each.

In [15]:
result, annotated, summary = run_amazon_categorization(
    input_csv_path=input_csv,
    state_dir=state_dir,
    output_dir=output_dir,
)

print(f'Categories known : {result.categories_count}')
print(f'Items mapped     : {result.mapped_items_count}')
print(f'Uncategorized    : {result.uncategorized_count}')

INFO analysis.amazon.categorizer: categorize.run.start
INFO analysis.amazon.categorizer: categorize.run.complete


Categories known : 2
Items mapped     : 3
Uncategorized    : 2


## Annotated output

In [16]:
annotated

,Description,Amount,normalized_item,category
0,Echo Dot (3rd Gen),24.99,echo dot (3rd gen),Electronics
1,Paper Towels 12-Pack,18.49,paper towels 12-pack,Household
2,USB-C Cable 6ft,9.99,usb-c cable 6ft,Electronics
3,Paper Towels 12-Pack,18.49,paper towels 12-pack,Household
4,Coffee Pods 36ct,28.00,coffee pods 36ct,UNCATEGORIZED
5,Unknown Gadget XYZ,45.00,unknown gadget xyz,UNCATEGORIZED


## Category summary

In [17]:
summary

,category,amount_total
0,UNCATEGORIZED,73.00
1,Household,36.98
2,Electronics,34.98


## Persistent state after run

Unknown items that were actively categorized this run are now remembered.
Items left as `UNCATEGORIZED` are intentionally excluded from the mapping
so they prompt again next month.

In [18]:
cats, mapping = load_categorization_state(state_dir)
print('Categories:', cats)
print('Mapping:')
for item, cat in sorted(mapping.items()):
    print(f'  {item!r:35} -> {cat}')

Categories: ['Electronics', 'Household']
Mapping:
  'echo dot (3rd gen)'                -> Electronics
  'paper towels 12-pack'              -> Household
  'usb-c cable 6ft'                   -> Electronics


## Structured run events

`events.jsonl` is only written when `configure_logging(state_dir)` is called
(done by the CLI).  Here we called `configure_logging()` with no path, so
the frame will be empty — swap in `state_dir` below to see events from real runs.

In [19]:
events = load_events_dataframe(state_dir)

if events.empty:
    print('No events.jsonl yet — call configure_logging(state_dir) to enable.')
else:
    display(events[['ts', 'event', 'row_count', 'uncategorized_count']])

No events.jsonl yet — call configure_logging(state_dir) to enable.


In [20]:
input_csv

PosixPath('/var/folders/35/5769v36d35d8hm4dykc4ll_r0000gn/T/amazon_playground_tbv7hs_1/amazon_jan.csv')

### Exploring event details

The `events` table records everything that happens in the system. Each event can include extra details in the `payload_json` column. This is like a bundle of extra info (for example: counts, filenames, or other context). You can easily explore these details in Python using pandas, even if you don't know SQL. AI assistants can help you extract or summarize this info if you want.

In [ ]:
import pandas as pd
from analysis.amazon.notebook import load_events_dataframe

events = load_events_dataframe(state_dir=state_dir)
print('Available columns:', events.columns)
events.head()  # Show the first few events

State directory: /var/folders/35/5769v36d35d8hm4dykc4ll_r0000gn/T/amazon_playground_iftgek85/state
Available columns: Index(['ts', 'level', 'logger', 'message', 'event', 'input_csv', 'row_count',
       'user', 'annotated_csv', 'summary_csv', 'uncategorized_count',
       'categories_count', 'mapped_items_count'],
      dtype='str')


,ts,level,logger,message,event,input_csv,row_count,user,annotated_csv,summary_csv,uncategorized_count,categories_count,mapped_items_count
0,2026-05-15T21:21:47,INFO,analysis.amazon.categorizer,categorize.run.start,categorize.run.start,/var/folders/35/5769v36d35d8hm4dykc4ll_r0000gn...,6,rwk,NaN,NaN,NaN,NaN,NaN
1,2026-05-15T21:21:47,INFO,analysis.amazon.categorizer,categorize.run.complete,categorize.run.complete,/var/folders/35/5769v36d35d8hm4dykc4ll_r0000gn...,6,rwk,/var/folders/35/5769v36d35d8hm4dykc4ll_r0000gn...,/var/folders/35/5769v36d35d8hm4dykc4ll_r0000gn...,2.0,2.0,3.0
